# EE/CS 148B HW1: Colab Setup

This notebook is meant to be imported directly into Colab to provide a quickstart environment setup.


## Colab Setup

Before running:

- Switch the runtime to a GPU runtime (T4 is fine; A100 is faster).
- Put the `hw1` directory somewhere accessible from Colab, typically Google Drive.

This notebook assumes the repo already exists and only sets up Python dependencies plus the repo import path.

Note: We will be using `pip` for dependencies inside Colab.

In [2]:
%%capture
!pip -q install -U regex jaxtyping einops psutil tiktoken pytest matplotlib pandas tqdm

In [1]:
from pathlib import Path

USE_DRIVE = True
DRIVE_REPO_ROOT = Path('/content/drive/MyDrive/hw1-main/')  # edit if needed
LOCAL_REPO_ROOT = Path('/content/hw1-sols')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = DRIVE_REPO_ROOT
else:
    REPO_ROOT = LOCAL_REPO_ROOT

assert REPO_ROOT.exists(), f'Repo root does not exist: {REPO_ROOT}'
print('Using repo:', REPO_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using repo: /content/drive/MyDrive/hw1-main


In [2]:
import os
import sys

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

cwd = /content/drive/MyDrive/hw1-main


In [3]:
import gc
import json
import math
import random
import subprocess
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    try:
        print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False).stdout)
    except FileNotFoundError:
        pass


device = cuda
gpu = Tesla T4
GPU 0: Tesla T4 (UUID: GPU-b7d1563d-429b-a17b-ed5a-27279b9b9d15)



# Your code starts here!

## 1. Sanity check: run pytest

Before training anything, verify that all unit tests pass.

In [23]:
!python -m pytest tests/ -x -q 2>&1 | tail -50

          8.6602e-01, -3.8866e-01]])
ffn_w2_weight = tensor([[-0.7916, -0.7535,  1.7878,  ..., -3.0037,  1.8461,  0.0519],
        [ 2.3077, -0.2597,  0.4373,  ..., -0.141...7035,  0.6021,  ..., -0.5654,  0.3101, -1.3532],
        [-1.9647, -0.4499, -0.9082,  ..., -2.3829, -1.2028,  1.3016]])
d_model = 64, d_ff = 128

    def test_ffn(numpy_snapshot, ffn_in_features, ffn_w1_weight, ffn_w2_weight, d_model, d_ff):
        actual_output = run_ffn(
            d_model=d_model,
            d_ff=d_ff,
            w1_weight=ffn_w1_weight,
            w2_weight=ffn_w2_weight,
            in_features=ffn_in_features,
        )
>       numpy_snapshot.assert_match(actual_output, atol=1e-6)

tests/test_model.py:48: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 
tests/conftest.py:90: in assert_match
    np.testing.assert_allclose(
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

args = (<function assert_allclose.<locals>.compare at

In [29]:
!python -m pytest tests/test_nn_utils.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/hw1-main
configfile: pyproject.toml
plugins: jaxtyping-0.3.10, langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collected 2 items                                                              

tests/test_nn_utils.py::test_softmax_matches_pytorch PASSED
tests/test_nn_utils.py::test_cross_entropy PASSED

============================== 2 passed in 0.11s ===============================


In [30]:
!python -m pytest tests/test_data.py -v


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/hw1-main
configfile: pyproject.toml
plugins: jaxtyping-0.3.10, langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collected 1 item                                                               

tests/test_data.py::test_get_batch PASSED

============================== 1 passed in 0.46s ===============================


In [5]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/hw1-main")

from eecs148b_hw1.bpe import train_bpe
from eecs148b_hw1.tokenizer import Tokenizer


In [6]:
!python -m pytest tests/test_nn_utils.py -v
!python -m pytest tests/test_data.py -v
!python -m pytest tests/test_model.py::test_layernorm tests/test_model.py::test_sinusoidal_pe -v
!python -m pytest tests/test_model.py -v
!python -m pytest tests/test_train_bpe.py -v
!python -m pytest tests/test_tokenizer.py -v


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/hw1-main
configfile: pyproject.toml
plugins: jaxtyping-0.3.10, langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collected 2 items                                                              

tests/test_nn_utils.py::test_softmax_matches_pytorch PASSED
tests/test_nn_utils.py::test_cross_entropy PASSED

============================== 2 passed in 0.08s ===============================
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/hw1-main
configfile: pyproject.toml
plugins: jaxtyping-0.3.10, langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collected 1 item                                                       

## 2. Download TinyStories

The dataset lives on HuggingFace; we cache it under `data/` in the repo so it's persisted on Drive.

In [7]:
DATA_DIR = REPO_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

TRAIN_TXT = DATA_DIR / 'TinyStoriesV2-GPT4-train.txt'
VAL_TXT   = DATA_DIR / 'TinyStoriesV2-GPT4-valid.txt'

if not TRAIN_TXT.exists():
    !cd {DATA_DIR} && wget -q https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-train.txt
if not VAL_TXT.exists():
    !cd {DATA_DIR} && wget -q https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStoriesV2-GPT4-valid.txt

!ls -lh {DATA_DIR}

total 2.1G
-rw------- 1 root root 2.1G May 30 03:16 TinyStoriesV2-GPT4-train.txt
-rw------- 1 root root  22M May 30 03:16 TinyStoriesV2-GPT4-valid.txt


## 3. Train the BPE tokenizer

`vocab_size=10_000` per spec.  Takes a few minutes on the full TinyStories train file.

Skips retraining if the saved tokenizer already exists on Drive.

In [9]:
from eecs148b_hw1.bpe import train_bpe
from eecs148b_hw1.tokenizer import Tokenizer

VOCAB_SIZE = 10_000
SPECIAL_TOKENS = ['<|endoftext|>']

VOCAB_PATH  = DATA_DIR / 'tokenizer_vocab.json'
MERGES_PATH = DATA_DIR / 'tokenizer_merges.txt'

if VOCAB_PATH.exists() and MERGES_PATH.exists():
    print('Tokenizer already exists — loading from disk.')
    tok = Tokenizer.from_files(VOCAB_PATH, MERGES_PATH, special_tokens=SPECIAL_TOKENS)
    vocab, merges = tok.vocab, tok.merges
else:
    t0 = time.time()
    vocab, merges = train_bpe(str(TRAIN_TXT), VOCAB_SIZE, SPECIAL_TOKENS)
    print(f'BPE training done in {time.time()-t0:.0f}s   ({len(vocab)} tokens, {len(merges)} merges)')
    tok = Tokenizer(vocab, merges, special_tokens=SPECIAL_TOKENS)
    tok.save_to_files(VOCAB_PATH, MERGES_PATH)
    print(f'Saved vocab → {VOCAB_PATH}')
    print(f'Saved merges → {MERGES_PATH}')


KeyboardInterrupt: 

## 4. Tokenizer analysis (TEXT DELIVERABLES)

Prints:
1. The longest BPE token learned
2. Compression ratio (bytes per token) on the validation set

In [12]:
# ----- LONGEST TOKEN -----
special_bytes = {t.encode('utf-8') for t in SPECIAL_TOKENS}
candidates = [(tid, b) for tid, b in vocab.items() if b not in special_bytes]
longest_id, longest_bytes = max(candidates, key=lambda kv: len(kv[1]))
try:
    longest_str = longest_bytes.decode('utf-8')
except UnicodeDecodeError:
    longest_str = repr(longest_bytes)
print(f'Longest BPE token: id={longest_id}   length={len(longest_bytes)} bytes   value={longest_str!r}')

# ----- COMPRESSION RATIO on validation set -----
MAX_BYTES = 5_000_000
with open(VAL_TXT, 'rb') as f:
    raw_bytes = f.read(MAX_BYTES)
text_sample = raw_bytes.decode('utf-8', errors='replace')
n_bytes = len(text_sample.encode('utf-8'))
ids = tok.encode(text_sample)
n_tokens = len(ids)
ratio = n_bytes / n_tokens
print(f'\nCompression ratio on {n_bytes:,} bytes of validation text:')
print(f'  tokens          = {n_tokens:,}')
print(f'  bytes per token = {ratio:.3f}')

NameError: name 'SPECIAL_TOKENS' is not defined

## 5. Tokenize train + val into .npy

Stored on Drive so we don't re-tokenize next time.

In [8]:
def tokenize_to_npy(txt_path, out_path):
    if Path(out_path).exists():
        arr = np.load(out_path, mmap_mode='r')
        print(f'  {out_path} already exists ({len(arr):,} tokens) — skipping.')
        return
    t0 = time.time()
    ids = []
    with open(txt_path, 'r', encoding='utf-8') as f:
        for tid in tok.encode_iterable(f):
            ids.append(tid)
    dtype = np.uint16 if len(tok.vocab) < 2**16 else np.int32
    arr = np.asarray(ids, dtype=dtype)
    np.save(out_path, arr)
    print(f'  {out_path}: {len(arr):,} tokens   ({time.time()-t0:.0f}s)')

TRAIN_IDS_PATH = DATA_DIR / 'train_ids.npy'
VAL_IDS_PATH   = DATA_DIR / 'val_ids.npy'

print('Tokenizing train ...')
tokenize_to_npy(TRAIN_TXT, TRAIN_IDS_PATH)
print('Tokenizing val ...')
tokenize_to_npy(VAL_TXT, VAL_IDS_PATH)

TRAIN_IDS = np.load(TRAIN_IDS_PATH, mmap_mode='r')
VAL_IDS   = np.load(VAL_IDS_PATH,   mmap_mode='r')
print(f'train: {len(TRAIN_IDS):,}   val: {len(VAL_IDS):,}')

Tokenizing train ...


NameError: name 'tok' is not defined

## 6. Training loop

Defines `train_run(...)` which builds a `TransformerLM`, trains it with AdamW + cosine LR schedule with linear warmup, gradient clipping, returns a list of log records and the trained model.

Defaults match the assignment spec:
- `vocab_size=10000`, `context_length=256`
- `d_model=512`, `num_layers=4`, `num_heads=8`, `d_ff=2048`
- total tokens ≈ 40.96 M → 2500 steps at batch_size=64

In [ ]:
from eecs148b_hw1.modules import TransformerLM
from eecs148b_hw1.nn_utils import cross_entropy
from eecs148b_hw1.data import get_batch

LOGS_DIR        = REPO_ROOT / 'logs'
CHECKPOINTS_DIR = REPO_ROOT / 'checkpoints'
FIGURES_DIR     = REPO_ROOT / 'figures'
LOGS_DIR.mkdir(exist_ok=True)
CHECKPOINTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)


def train_run(
    run_name,
    *,
    total_tokens=40_960_000,
    batch_size=64,
    context_length=256,
    vocab_size=10_000,
    d_model=512, num_layers=4, num_heads=8, d_ff=2048,
    learning_rate=3e-4, warmup_steps=200, weight_decay=0.01,
    beta1=0.9, beta2=0.95, eps=1e-8, grad_clip=1.0,
    use_layernorm=True, use_pos_embedding=True,
    eval_interval=250, eval_iters=20, log_interval=50,
    seed=42,
    early_stop_val_loss=None,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = TransformerLM(
        vocab_size=vocab_size,
        context_length=context_length,
        d_model=d_model,
        num_layers=num_layers,
        num_heads=num_heads,
        d_ff=d_ff,
        use_pos_embedding=use_pos_embedding,
        use_layernorm=use_layernorm,
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters())
    print(f'[{run_name}] {n_params/1e6:.1f}M params  '
          f'use_layernorm={use_layernorm}  use_pos_embedding={use_pos_embedding}  lr={learning_rate}')

    optim = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate, weight_decay=weight_decay, betas=(beta1, beta2), eps=eps,
    )

    tokens_per_step = batch_size * context_length
    total_steps = total_tokens // tokens_per_step
    print(f'[{run_name}] total_steps={total_steps}  tokens/step={tokens_per_step}')

    def lr_at(step):
        if step < warmup_steps:
            return learning_rate * (step + 1) / warmup_steps
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        return learning_rate * 0.5 * (1.0 + math.cos(math.pi * progress))

    records = []
    t0 = time.time()
    model.train()
    diverged = False

    for step in range(total_steps):
        lr = lr_at(step)
        for pg in optim.param_groups:
            pg['lr'] = lr

        x, y = get_batch(TRAIN_IDS, batch_size, context_length, str(DEVICE))
        logits = model(x)
        loss = cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optim.step()

        loss_val = loss.item()
        if not math.isfinite(loss_val):
            print(f'[{run_name}] step={step} train_loss is {loss_val} — stopping (diverged).')
            diverged = True
            break

        if step % log_interval == 0 or step == total_steps - 1:
            records.append({'step': step, 'train_loss': loss_val,
                            'lr': lr, 'elapsed': time.time()-t0})

        if step % eval_interval == 0 or step == total_steps - 1:
            model.eval()
            losses = []
            with torch.no_grad():
                for _ in range(eval_iters):
                    xv, yv = get_batch(VAL_IDS, batch_size, context_length, str(DEVICE))
                    logits_v = model(xv)
                    losses.append(cross_entropy(
                        logits_v.reshape(-1, logits_v.size(-1)),
                        yv.reshape(-1)).item())
            val_loss = float(np.mean(losses))
            records.append({'step': step, 'val_loss': val_loss,
                            'elapsed': time.time()-t0})
            print(f'[{run_name}] step={step:5d}/{total_steps}  '
                  f'train_loss={loss_val:.3f}  val_loss={val_loss:.3f}  '
                  f'lr={lr:.5f}  t={time.time()-t0:.0f}s')
            model.train()
            if early_stop_val_loss is not None and val_loss <= early_stop_val_loss:
                print(f'[{run_name}] reached val_loss <= {early_stop_val_loss} -- stopping early.')
                break

    print(f'[{run_name}] done in {time.time()-t0:.0f}s  (diverged={diverged})')
    return model, records


def save_log(records, path):
    with open(path, 'w') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')
    print(f'  log saved → {path}')

## 7. Run the four training experiments

> **Time budget:** roughly 12-18 min per run on T4, faster on A100. Four runs total ≈ 1-1.5h on T4.
>
> Each `train_run(...)` call returns `(model, log_records)`.  Logs are saved to Drive so you can re-plot without retraining.

### 7.1 Baseline — Figure 1 (train until `val_loss ≤ 2.0`)

In [ ]:
model_base, log_base = train_run(
    'baseline',
    total_tokens=40_960_000,
    learning_rate=3e-4,
    warmup_steps=200,
    use_layernorm=True,
    use_pos_embedding=True,
    early_stop_val_loss=2.0,   # stop early once we hit the spec target
)
save_log(log_base, LOGS_DIR / 'baseline.jsonl')

# Save baseline checkpoint for generation later.
torch.save({
    'model_state': model_base.state_dict(),
    'config': dict(vocab_size=10_000, context_length=256, d_model=512,
                   num_layers=4, num_heads=8, d_ff=2048,
                   use_layernorm=True, use_pos_embedding=True),
}, CHECKPOINTS_DIR / 'baseline.pt')
print('Saved checkpoint → checkpoints/baseline.pt')

# Free up VRAM for the next run.
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

### 7.2 No LayerNorm at same LR — Figure 2

Same LR as baseline (3e-4). Expect this to diverge or train poorly without LayerNorm.

In [ ]:
model_noln_samelr, log_noln_samelr = train_run(
    'no-layernorm-same-lr',
    total_tokens=40_960_000,
    learning_rate=3e-4,
    warmup_steps=200,
    use_layernorm=False,
    use_pos_embedding=True,
)
save_log(log_noln_samelr, LOGS_DIR / 'no_layernorm_same_lr.jsonl')

del model_noln_samelr
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

### 7.3 No LayerNorm with lower LR — Figure 3

Drop the LR 10× to see if the no-LN model can train at all with a more conservative LR.

In [ ]:
model_noln_lowlr, log_noln_lowlr = train_run(
    'no-layernorm-low-lr',
    total_tokens=40_960_000,
    learning_rate=3e-5,
    warmup_steps=200,
    use_layernorm=False,
    use_pos_embedding=True,
)
save_log(log_noln_lowlr, LOGS_DIR / 'no_layernorm_low_lr.jsonl')

del model_noln_lowlr
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

### 7.4 NoPE — Figure 4

Same as baseline but with no positional encoding at all.

In [ ]:
model_nope, log_nope = train_run(
    'no-pos-embedding',
    total_tokens=40_960_000,
    learning_rate=3e-4,
    warmup_steps=200,
    use_layernorm=True,
    use_pos_embedding=False,
)
save_log(log_nope, LOGS_DIR / 'no_pos_embedding.jsonl')

del model_nope
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 8. Plot the four required figures

In [ ]:
def split_records(records):
    train_x, train_y = [], []
    val_x, val_y = [], []
    for r in records:
        if 'train_loss' in r:
            train_x.append(r['step']); train_y.append(r['train_loss'])
        if 'val_loss' in r:
            val_x.append(r['step']); val_y.append(r['val_loss'])
    return train_x, train_y, val_x, val_y

def plot_curves(runs, title, out_path, ylim=None):
    """runs = list of (label, records, color)."""
    fig, ax = plt.subplots(figsize=(8, 5))
    for label, records, color in runs:
        tx, ty, vx, vy = split_records(records)
        ax.plot(tx, ty, alpha=0.4, color=color, label=f'{label} (train)')
        if vy:
            ax.plot(vx, vy, marker='o', linestyle='--', color=color,
                    label=f'{label} (val)')
    ax.set_xlabel('step')
    ax.set_ylabel('cross-entropy loss')
    ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    print('Saved →', out_path)

# Figure 1
plot_curves(
    [('baseline', log_base, 'C0')],
    'Figure 1: Baseline training (val_loss target ≤ 2.0)',
    FIGURES_DIR / 'figure1_baseline.png',
)

In [ ]:
# Figure 2
plot_curves(
    [('baseline (with LayerNorm)', log_base, 'C0'),
     ('no LayerNorm (same LR=3e-4)', log_noln_samelr, 'C3')],
    'Figure 2: LayerNorm ablation — same LR',
    FIGURES_DIR / 'figure2_layernorm_same_lr.png',
)

In [ ]:
# Figure 3
plot_curves(
    [('baseline (with LayerNorm, LR=3e-4)', log_base, 'C0'),
     ('no LayerNorm (LR=3e-5)', log_noln_lowlr, 'C2')],
    'Figure 3: LayerNorm ablation — lower LR',
    FIGURES_DIR / 'figure3_layernorm_low_lr.png',
)

In [ ]:
# Figure 4
plot_curves(
    [('sinusoidal PE (baseline)', log_base, 'C0'),
     ('NoPE (no positional encoding)', log_nope, 'C4')],
    'Figure 4: NoPE vs. Sinusoidal positional encoding',
    FIGURES_DIR / 'figure4_nope_vs_sin.png',
)

## 9. Generate 300 tokens from the baseline

Temperature 0.8, top-p 0.9.  Saves to `generated_sample.txt` on Drive.

In [ ]:
from eecs148b_hw1.nn_utils import softmax

@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=300,
             temperature=0.8, top_p=0.9, eos_token='<|endoftext|>', device='cuda'):
    model.eval()
    tokens = list(tokenizer.encode(prompt))
    eos_id = tokenizer.special_to_id.get(eos_token)
    context_length = model.context_length

    for _ in range(max_new_tokens):
        context = tokens[-context_length:]
        x = torch.tensor(context, dtype=torch.long, device=device).unsqueeze(0)
        logits = model(x)[0, -1]

        if temperature <= 0:
            nxt = int(torch.argmax(logits).item())
        else:
            probs = softmax(logits / temperature, dim=-1)
            if 0.0 < top_p < 1.0:
                sorted_p, sorted_idx = torch.sort(probs, descending=True)
                cumulative = torch.cumsum(sorted_p, dim=-1)
                keep = cumulative - sorted_p < top_p
                kept = torch.where(keep, sorted_p, torch.zeros_like(sorted_p))
                kept = kept / kept.sum()
                choice = int(torch.multinomial(kept, num_samples=1).item())
                nxt = int(sorted_idx[choice].item())
            else:
                nxt = int(torch.multinomial(probs, num_samples=1).item())

        tokens.append(nxt)
        if eos_id is not None and nxt == eos_id:
            break

    return tokenizer.decode(tokens)


PROMPT = 'Once upon a time'
sample = generate(model_base, tok, PROMPT,
                  max_new_tokens=300, temperature=0.8, top_p=0.9,
                  device=str(DEVICE))
print('==== GENERATED SAMPLE (max 300 tokens) ====\n')
print(sample)
print('\n===========================================')

with open(REPO_ROOT / 'generated_sample.txt', 'w') as f:
    f.write(sample)
print(f'\nSaved → {REPO_ROOT / "generated_sample.txt"}')